# 01 — Exploratory Data Analysis
**Project:** AI-Generated vs Real Human Face Detection  
**Dataset:** 140k Real and Fake Faces (Kaggle: xhlulu/140k-real-and-fake-faces)  

This notebook covers:
- **Week 3:** Descriptive Statistics (mean, median, std, skewness, kurtosis, correlation)
- **Week 4:** Data Preprocessing & Cleaning
- **Week 6:** Data Visualization (histogram, boxplot, scatter plot, heatmap)
- **Week 7:** Hypothesis Testing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
from scipy import stats

PROCESSED = Path('../data/processed')
train_df = pd.read_csv(PROCESSED / 'train.csv')
val_df   = pd.read_csv(PROCESSED / 'val.csv')
test_df  = pd.read_csv(PROCESSED / 'test.csv')

print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))

---
## Section 4 — Dataset Description
*(Applies Week 3: understanding data structure and features)*

In [ ]:
print('=== Dataset Description ===')
print(f'Source         : Kaggle — xhlulu/140k-real-and-fake-faces')
print(f'Real images    : Flickr-Faces-HQ (FFHQ) — real human portraits')
print(f'Fake images    : StyleGAN-generated synthetic faces')
print(f'Total records  : {len(train_df) + len(val_df) + len(test_df)}')
print(f'Features       : filepath (str), label (int: 0=REAL, 1=AI_GENERATED)')
print(f'\nSplit sizes:')
print(f'  Train : {len(train_df)}')
print(f'  Val   : {len(val_df)}')
print(f'  Test  : {len(test_df)}')
print(f'\nLabel distribution (Train):')
print(train_df['label'].value_counts().rename({0: 'REAL', 1: 'AI_GENERATED'}))

---
## Section 5 — Data Preprocessing & Cleaning
*(Applies Week 4: handling missing values, duplicates, data validation)*

In [ ]:
print('=== Step 1: Check for Missing Values ===')
print(train_df.isnull().sum())
print(f'\nMissing values in train: {train_df.isnull().sum().sum()}')
print(f'Missing values in val  : {val_df.isnull().sum().sum()}')
print(f'Missing values in test : {test_df.isnull().sum().sum()}')

In [ ]:
print('=== Step 2: Check for Duplicate File Paths ===')
print(f'Duplicate paths in train : {train_df["filepath"].duplicated().sum()}')
print(f'Duplicate paths in val   : {val_df["filepath"].duplicated().sum()}')
print(f'Duplicate paths in test  : {test_df["filepath"].duplicated().sum()}')

In [ ]:
print('=== Step 3: Check for Corrupted / Unreadable Images ===')
bad_files = []
sample_check = train_df.sample(min(300, len(train_df)), random_state=42)

for path in sample_check['filepath']:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        bad_files.append(path)

print(f'Corrupted images in sample of {len(sample_check)}: {len(bad_files)}')
if bad_files:
    print('Corrupted files:', bad_files[:5])

In [ ]:
print('=== Step 4: Verify Image Sizes ===')
sample = train_df.sample(min(300, len(train_df)), random_state=42)
widths, heights = [], []

for path in sample['filepath']:
    try:
        with Image.open(path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except:
        pass

print(f'Width  — mean: {np.mean(widths):.0f}, min: {min(widths)}, max: {max(widths)}')
print(f'Height — mean: {np.mean(heights):.0f}, min: {min(heights)}, max: {max(heights)}')
print(f'\nAll images same size: {len(set(zip(widths, heights))) == 1}')
print('\nPreprocessing steps applied during training:')
print('  - Resize to 224x224 (EfficientNet input requirement)')
print('  - Normalize pixels to [0,1]')
print('  - Apply ImageNet mean/std normalization')
print('  - Data augmentation: RandomHorizontalFlip, RandomRotation, ColorJitter')

---
## Section 6 — Exploratory Data Analysis (EDA)
*(Applies Week 3: Descriptive Statistics + Week 6: Data Visualization)*

### 6a. Sample Images — REAL vs AI-GENERATED

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
label_names = {0: 'REAL', 1: 'AI_GENERATED'}
colors = {0: '#22c55e', 1: '#ef4444'}

for i, label in enumerate([0, 1]):
    samples = train_df[train_df['label'] == label].sample(5, random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        img = mpimg.imread(row['filepath'])
        axes[i][j].imshow(img)
        axes[i][j].axis('off')
        if j == 0:
            axes[i][j].set_ylabel(label_names[label], fontsize=12,
                                   color=colors[label], fontweight='bold',
                                   rotation=0, labelpad=60)

plt.suptitle('Sample Images: Real Human Faces vs AI-Generated Faces', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()
print('Insight: AI-generated faces may look realistic but often have subtle artifacts in hair, teeth, and background edges.')

### 6b. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (df, name) in zip(axes, [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar(['REAL (0)', 'AI (1)'], counts.values, color=['#22c55e', '#ef4444'], width=0.5)
    ax.set_title(f'{name} Set — {len(df)} images', fontsize=12)
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 50, str(v), ha='center', fontsize=10)

plt.suptitle('Class Distribution Across Splits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print('Insight: The dataset is perfectly balanced — equal number of REAL and AI images in each split.')
print('This means we do NOT need to apply class weighting or oversampling.')

### 6c. Extract Image Pixel Features for Statistical Analysis
*(Week 3 & Week 9: Feature Engineering — extracting numerical features from images)*

In [ ]:
def extract_image_stats(filepath):
    """Extract pixel-level statistics from an image for EDA."""
    img = cv2.imread(filepath)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    return {
        'mean_brightness' : img_rgb.mean(),
        'std_brightness'  : img_rgb.std(),
        'mean_red'        : img_rgb[:,:,0].mean(),
        'mean_green'      : img_rgb[:,:,1].mean(),
        'mean_blue'       : img_rgb[:,:,2].mean(),
        'contrast'        : gray.std(),
    }

# Sample 500 per class for speed
real_paths = train_df[train_df['label']==0].sample(500, random_state=42)['filepath']
fake_paths = train_df[train_df['label']==1].sample(500, random_state=42)['filepath']

real_stats = pd.DataFrame([extract_image_stats(p) for p in real_paths]).dropna()
fake_stats = pd.DataFrame([extract_image_stats(p) for p in fake_paths]).dropna()
real_stats['label'] = 0
fake_stats['label'] = 1
combined = pd.concat([real_stats, fake_stats], ignore_index=True)

print(f'Features extracted per image: {list(real_stats.columns[:-1])}')
print(f'Sample size: {len(real_stats)} REAL + {len(fake_stats)} AI')

### 6d. Descriptive Statistics
*(Week 3: mean, median, std, min, max, quartiles)*

In [ ]:
print('=== REAL Images — Descriptive Statistics ===')
print(real_stats.drop('label', axis=1).describe().round(2))

print('\n=== AI-GENERATED Images — Descriptive Statistics ===')
print(fake_stats.drop('label', axis=1).describe().round(2))

In [ ]:
# Week 3: Skewness and Kurtosis
print('=== Skewness & Kurtosis (mean_brightness) ===')
print(f'REAL  — Skewness: {real_stats["mean_brightness"].skew():.4f}')
print(f'REAL  — Kurtosis: {real_stats["mean_brightness"].kurtosis():.4f}')
print(f'AI    — Skewness: {fake_stats["mean_brightness"].skew():.4f}')
print(f'AI    — Kurtosis: {fake_stats["mean_brightness"].kurtosis():.4f}')
print('\nInterpretation:')
print('  Skewness > 0 = right-skewed (more dark images)')
print('  Skewness < 0 = left-skewed (more bright images)')
print('  Kurtosis > 0 = more extreme values (outliers) than normal distribution')

### 6e. Histogram — Brightness Distribution
*(Week 6: Data Visualization)*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(real_stats['mean_brightness'], bins=40, color='#22c55e', alpha=0.7, label='REAL')
axes[0].hist(fake_stats['mean_brightness'], bins=40, color='#ef4444', alpha=0.7, label='AI_GENERATED')
axes[0].axvline(real_stats['mean_brightness'].mean(), color='green', linestyle='--', label=f'REAL mean={real_stats["mean_brightness"].mean():.1f}')
axes[0].axvline(fake_stats['mean_brightness'].mean(), color='red',   linestyle='--', label=f'AI mean={fake_stats["mean_brightness"].mean():.1f}')
axes[0].set_title('Brightness Distribution (Histogram)')
axes[0].set_xlabel('Mean Pixel Brightness')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Boxplot
data_to_plot = [real_stats['mean_brightness'].values, fake_stats['mean_brightness'].values]
bp = axes[1].boxplot(data_to_plot, labels=['REAL', 'AI_GENERATED'],
                     patch_artist=True,
                     boxprops=dict(facecolor='#bbf7d0'),
                     medianprops=dict(color='black', linewidth=2))
bp['boxes'][1].set_facecolor('#fecaca')
axes[1].set_title('Brightness Distribution (Boxplot)')
axes[1].set_ylabel('Mean Pixel Brightness')

plt.suptitle('Image Brightness: Real vs AI-Generated Faces', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('brightness_distribution.png', dpi=150)
plt.show()
print('Insight: Compare means and spread. If AI images have different brightness patterns, the model can use this as a signal.')

### 6f. Color Channel Comparison
*(Week 6: Multi-panel Visualization)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
channels = ['mean_red', 'mean_green', 'mean_blue']
ch_colors = ['#ef4444', '#22c55e', '#3b82f6']
titles    = ['Red Channel', 'Green Channel', 'Blue Channel']

for ax, ch, color, title in zip(axes, channels, ch_colors, titles):
    ax.hist(real_stats[ch], bins=35, alpha=0.6, color='gray',  label='REAL')
    ax.hist(fake_stats[ch], bins=35, alpha=0.6, color=color,   label='AI')
    ax.axvline(real_stats[ch].mean(), color='black', linestyle='--', linewidth=1.5)
    ax.axvline(fake_stats[ch].mean(), color=color,   linestyle='-',  linewidth=1.5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Mean Pixel Value')
    ax.legend()

plt.suptitle('RGB Channel Distribution: Real vs AI Faces', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('color_channels.png', dpi=150)
plt.show()
print('Insight: StyleGAN-generated faces tend to have slightly different color temperature than real photographs.')

### 6g. Scatter Plot — Brightness vs Contrast
*(Week 6: Scatter Plot)*

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(real_stats['mean_brightness'], real_stats['contrast'],
            alpha=0.4, color='#22c55e', label='REAL', s=15)
plt.scatter(fake_stats['mean_brightness'], fake_stats['contrast'],
            alpha=0.4, color='#ef4444', label='AI_GENERATED', s=15)
plt.xlabel('Mean Brightness')
plt.ylabel('Contrast (Std of Grayscale)')
plt.title('Brightness vs Contrast: Real vs AI Faces', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('scatter_brightness_contrast.png', dpi=150)
plt.show()
print('Insight: If clusters are separable, handcrafted features alone could partially classify images.')
print('If clusters overlap heavily, deep learning is necessary — which is why we use EfficientNet.')

### 6h. Correlation Heatmap
*(Week 3: Correlation analysis)*

In [ ]:
plt.figure(figsize=(9, 7))
corr = combined.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle duplicates
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap of Image Features + Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('Insight: Check correlation between each feature and the label (last column).')
print('Strong correlation means that feature is a good predictor of real vs AI.')

---
## Section 7 — Hypothesis Testing
*(Week 7: Statistical Hypothesis Testing using t-test)*

In [ ]:
print('=' * 60)
print('HYPOTHESIS TESTING')
print('=' * 60)
print()

hypotheses = [
    ('H1', 'mean_brightness', 'AI-generated faces have different mean brightness than real faces'),
    ('H2', 'contrast',        'AI-generated faces have different contrast than real faces'),
    ('H3', 'mean_red',        'AI-generated faces have different red channel values than real faces'),
    ('H4', 'std_brightness',  'AI-generated faces have different pixel variance than real faces'),
]

alpha = 0.05
for name, feature, description in hypotheses:
    t_stat, p_value = stats.ttest_ind(
        real_stats[feature],
        fake_stats[feature],
        equal_var=False  # Welch's t-test (safer when variance differs)
    )
    result = 'REJECT H0 — Significant difference found' if p_value < alpha else 'FAIL to reject H0 — No significant difference'
    print(f'{name}: {description}')
    print(f'  t-statistic : {t_stat:.4f}')
    print(f'  p-value     : {p_value:.6f}')
    print(f'  Result      : {result} (α = {alpha})')
    print()

In [ ]:
# Visual comparison of hypothesis features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = ['mean_brightness', 'contrast', 'mean_red', 'std_brightness']
h_labels = ['H1: Mean Brightness', 'H2: Contrast', 'H3: Red Channel', 'H4: Pixel Variance']

for ax, feat, h_label in zip(axes.flat, features, h_labels):
    data = [real_stats[feat].values, fake_stats[feat].values]
    bp = ax.boxplot(data, labels=['REAL', 'AI_GENERATED'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#bbf7d0')
    bp['boxes'][1].set_facecolor('#fecaca')
    ax.set_title(h_label, fontsize=11, fontweight='bold')
    ax.set_ylabel(feat)

plt.suptitle('Hypothesis Testing — Feature Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hypothesis_boxplots.png', dpi=150)
plt.show()

---
## Summary of EDA Findings

Write your interpretation here after running all cells. Example:

1. **Class balance**: The dataset is perfectly balanced (50% REAL / 50% AI) — no class imbalance issue.
2. **Brightness**: AI images tend to have [higher/lower] mean brightness than real photos (see histogram).
3. **Contrast**: Real photos show [higher/lower] contrast due to natural lighting variation.
4. **Hypothesis results**: H1 and H2 [were/were not] rejected at α=0.05, meaning brightness and contrast [do/do not] differ significantly between real and AI faces.
5. **Correlation**: The feature most correlated with label is `[feature_name]` (r = X.XX).
6. **Conclusion**: Handcrafted pixel features show partial separability, but significant overlap remains — justifying the use of a deep learning model (EfficientNet-B0) for better classification.

**Next step → open `02b_improved_training.ipynb` to train the model.**